In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import uuid  # Added for generating unique IDs
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from google.colab import drive
import kagglehub
import random

# Configuration
DATASET_NAME = "mikeortman/wikipedia-sentences"
BATCH_SIZE = 256
MODEL_NAME = "all-MiniLM-L6-v2"
OUTPUT_DIR = "/content/drive/MyDrive/wikipedia_embeddings"

def download_dataset():
    """Download the Wikipedia sentences dataset using kagglehub"""
    print(f"\nDownloading dataset: {DATASET_NAME}")
    dataset_path = kagglehub.dataset_download(DATASET_NAME)
    print(f"✓ Dataset downloaded to: {dataset_path}")
    return dataset_path

def load_data(dataset_path):
    """Load the Wikipedia sentences"""
    print("\nLoading sentences...")
    file_path = None
    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if f.lower().endswith('.txt'):
                file_path = os.path.join(root, f)

    if not file_path:
        raise FileNotFoundError(f"No TXT file found in {dataset_path}")

    with open(file_path, "r") as f:
        sentences = [i.strip() for i in f.readlines()]

    # NOTE: Keeping your slice for testing, remove [:100] for full run

    random.shuffle(sentences)
    sentences = sentences[:2_000_000]
    print(f"✓ Loaded {len(sentences)} sentences")
    return sentences

def compute_embeddings(sentences, model, batch_size=BATCH_SIZE):
    """Compute embeddings for all sentences"""
    print(f"\nComputing embeddings with batch size {batch_size}...")

    embeddings = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Processing batches"):
        batch = sentences[i:i + batch_size]
        batch_embeddings = model.encode(
            batch,
            convert_to_numpy=True,
            show_progress_bar=False,
            batch_size=batch_size
        )
        embeddings.append(batch_embeddings)

    embeddings = np.vstack(embeddings)
    print(f"✓ Computed embeddings: {embeddings.shape}")
    return embeddings

def save_results_parquet(df, embeddings, output_dir=OUTPUT_DIR):
    """
    Save ID, Sentence, and Embedding in a single Parquet file.
    """
    os.makedirs(output_dir, exist_ok=True)
    print(f"\nSaving results to {output_dir}...")

    # 1. Generate Unique IDs (UUIDs are safer than index numbers for merging later)
    print("Generating unique IDs...")
    df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]

    # 2. Add Embeddings to DataFrame
    # Note: We convert the numpy matrix (N, 384) into a list of arrays
    # so pandas treats each row as a single object (the vector)
    print("Merging embeddings into DataFrame...")
    df['embedding'] = list(embeddings)

    # 3. Reorder columns for clarity
    df = df[['id', 'sentence', 'embedding']]

    # 4. Save to Parquet
    output_path = os.path.join(output_dir, "wikipedia_vectors.parquet")

    # engine='pyarrow' is standard for Colab
    df.to_parquet(output_path, index=False, engine='pyarrow')

    print(f"✓ Saved Parquet file to: {output_path}")

    # Save configuration info separately (optional, but good practice)
    config = {
        "model_name": MODEL_NAME,
        "num_sentences": len(df),
        "embedding_dim": embeddings.shape[1],
        "columns": list(df.columns)
    }
    with open(os.path.join(output_dir, "config.json"), 'w') as f:
        json.dump(config, f, indent=2)

    print(f"\n✓ Process Complete. File size: {os.path.getsize(output_path) / (1024**2):.2f} MB")

def main():
    print("=" * 60)
    print("Wikipedia Sentence Embeddings Generator (Parquet Edition)")
    print("=" * 60)

    drive.mount('/content/drive')

    dataset_path = download_dataset()
    sentences_list = load_data(dataset_path)

    # Create DataFrame immediately
    df = pd.DataFrame({'sentence': sentences_list})

    # Load Model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDevice: {device}")
    model = SentenceTransformer(MODEL_NAME, device=device)

    # Compute
    embeddings = compute_embeddings(sentences_list, model)

    # Save using the new Parquet function
    save_results_parquet(df, embeddings)

if __name__ == "__main__":
    main()

Wikipedia Sentence Embeddings Generator (Parquet Edition)
Mounted at /content/drive



100%|██████████| 314M/314M [00:14<00:00, 22.1MB/s]

Extracting files...


✓ Dataset downloaded to: /root/.cache/kagglehub/datasets/mikeortman/wikipedia-sentences/versions/3

Loading sentences...
✓ Loaded 2000000 sentences

Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Computing embeddings with batch size 256...


Processing batches:   0%|          | 0/7813 [00:00<?, ?it/s]

✓ Computed embeddings: (2000000, 384)

Saving results to /content/drive/MyDrive/wikipedia_embeddings...
Generating unique IDs...
Merging embeddings into DataFrame...


In [ ]:

# Load the file
df = pd.read_parquet("/content/drive/MyDrive/wikipedia_embeddings/wikipedia_vectors.parquet")

# Access data
print(df.iloc[0]['sentence'])
print(df.iloc[0]['embedding']) # This is already a numpy array/list

# If you need the whole matrix back for similarity search:
# Convert the column of arrays back to a 2D matrix
embedding_matrix = np.vstack(df['embedding'].values)